# Full phase outline for Paper 2 - Set Encoder

In [ ]:
# Cell 1: Setup
%load_ext autoreload
%autoreload 2

In [ ]:
import sys
from pathlib import Path
import numpy as np
import matplotlib.pyplot as plt

from storm_utils.config_paths import get_project_paths
from storm_regression.results_io import load_results, recreate_dataset_from_results
from storm_regression.plotting import plot_case_study, plot_ensemble_comparison_from_files, plot_window_data
from storm_regression.forecast_analysis import auto_detect_and_compare, auto_detect_and_plot_case_studies, analyze_results
import logging

paths = get_project_paths()
logging.disable(logging.INFO)

## Examine absolute skill (taking median of distribution)

In [ ]:
from storm_regression.forecast_analysis import create_comparison_dashboard

# Usage example:
results_file = paths['regression_results'] / 'HUXt3' / 'phase1_baseline' / 'results_seed42_lt1_fold0_thresh4.5_balanced_Bz_GSM_XGBoost_Fast.pkl'

comparison_df = create_comparison_dashboard(
    results_path=results_file,
    prediction_type='y_pred_lognormal_median',
    save=True,
    save_name='dashboard_comparison.png'
)

In [ ]:
from storm_regression.forecast_analysis import create_comparison_dashboard

# Usage example:
results_file = paths['regression_results'] / 'HUXt3' / 'phase1_baseline' / 'results_seed42_lt24_fold0_thresh4.5_balanced_Bz_GSM_MLP_stats.pkl'

comparison_df = create_comparison_dashboard(
    results_path=results_file,
    prediction_type='y_pred_lognormal_median',
    save=True,
    save_name='dashboard_comparison.png'
)

## Examine probabilistic skill
Baseline probabilistic verification of the initial distributional forecasts —
establishing the diagnostic toolkit (CRPS, PIT calibration, exceedance
reliability) that all subsequent phases are judged by. Confirms the evaluation
pipeline runs end-to-end before any model comparison.

In [ ]:
from storm_regression.forecast_analysis import create_probabilistic_dashboard

filepath = paths['regression_results'] / 'HUXt3' / 'results_seed42_lt24_fold0_thresh4.5_balanced_Bz_GSM_XGBoost_Fast.pkl'
probabilistic_df = create_probabilistic_dashboard(filepath)

In [ ]:
from storm_regression.forecast_analysis import create_probabilistic_dashboard

filepath = paths['regression_results'] / 'HUXt3' / 'results_seed42_lt24_fold0_thresh4.5_balanced_Bz_GSM_MLP_stats.pkl'
probabilistic_df = create_probabilistic_dashboard(filepath)

# Phase 1 — Baseline

This just includes all of the work on translating the methods of Billcliff et al. (2026) to a regression format, targeting $\text{Hp}30_\text{max,24h}$

In [ ]:
from storm_regression.forecast_analysis import create_comparison_dashboard

# Usage example:
results_file = paths['regression_results'] / 'HUXt3' / 'phase1_baseline'
results_file = results_file / 'results_seed42_lt24_fold0_thresh4.5_balanced_Bz_GSM_XGBoost_fast.pkl'

comparison_df = create_comparison_dashboard(
    results_path=results_file,
    prediction_type='y_pred_lognormal_median',
    save=True,
    save_name='dashboard_comparison.png'
)

In [ ]:
results_dir = paths['regression_results'] / 'HUXt3' / 'phase1_baseline'

results_df = auto_detect_and_compare(
    results_dir=results_dir,
    save=True,
    save_dir=Path('analysis_outputs')
)

# Phase 1 — Feature / ensemble representation

**Question:** does giving the MLP richer ensemble detail (more percentiles)
improve skill over the median alone?

**Setup:** flat MLP, sweeping percentile density (median → sparse → p5s → p1s),
12h, fold-robust across 5 folds.

**Finding:** density does NOT help. Dense representations (p1s, 99 percentiles)
actively hurt (~2σ worse, overfitting); median/sparse/p5s are statistical ties.
Confirmed across folds — not a fold-0 artefact.

**Decision:** the flat MLP cannot exploit ensemble detail; median suffices.
Representation is immaterial at this architecture. → motivates asking whether
the limit is representational (set encoder) or informational.

In [ ]:
from storm_regression.forecast_analysis import create_comparison_dashboard

# Usage example:
results_file = paths['regression_results'] / 'HUXt3' / 'phase1_features'
results_file = results_file / 'results_seed42_lt12_fold0_thresh4.5_balanced_Bz_GSM-Dst-n_sw_MLP_stats.pkl'

comparison_df = create_comparison_dashboard(
    results_path=results_file,
    prediction_type='y_pred_lognormal_median',
    save=True,
    save_name='dashboard_comparison.png'
)

In [ ]:
results_dir = paths['regression_results'] / 'HUXt3' / 'phase1_features'

results_df = auto_detect_and_compare(
    results_dir=results_dir,
    save=True,
    save_dir=Path('analysis_outputs')
)

In [ ]:
# Cell 3: Load and analyze specific result

results_folder = paths['regression_results'] / 'HUXt3' / 'phase1_features'
filepath = results_folder / 'results_seed42_lt12_fold0_thresh4.5_balanced_flow_pressure_MLP_stats.pkl'

case_study = analyze_results(
    filepath,
    n=10,
    event_types=None,
    descending=True,
    plot_function=plot_case_study,
)

# Phase 1 — Ensemble spread
**Question:** does explicitly adding spread features (std, IQR, range) help
beyond the percentiles?

**Finding:** dead heat — spread adds nothing (within noise across folds; if
anything marginally worse on storms).

**Decision:** drop `include_ensemble_spread`; it does not earn its place.

In [ ]:
from storm_regression.forecast_analysis import create_comparison_dashboard

# Usage example:
results_file = paths['regression_results'] / 'HUXt3' / 'phase1_ensemble_spread'
results_file = results_file / 'results_seed42_lt12_fold0_thresh4.5_balanced__MLP_flat_p5-25-50-75-95.pkl'

comparison_df = create_comparison_dashboard(
    results_path=results_file,
    prediction_type='y_pred_lognormal_median',
    save=True,
    save_name='dashboard_comparison.png'
)

In [ ]:
results_dir = paths['regression_results'] / 'HUXt3' / 'phase1_ensemble_spread'

results_df = auto_detect_and_compare(
    results_dir=results_dir,
    save=True,
    save_dir=Path('analysis_outputs')
)

# Phase 2 — Architecture

**Question:** can any network shape (depth/width) let the dense representation
beat the median — i.e. can architecture rescue what features could not?

**Setup:** 6 architectures × {median, p5s} inputs, K=1, 12h, fold 0; winner
fold-validated across 5 folds.

**Finding:** median beats or ties p5s at EVERY architecture (5 wins / 1 tie /
0 p5s-wins on storms). No architecture closes the gap. Deeper/wider helps the
median model modestly but adds nothing for percentiles. deep_wide's single-fold
edge did not survive fold-averaging (over-parameterised); deep_narrow held up.

**Decision:** locked deep_narrow [64,64,64,64]. Architecture does not break the
ceiling — strengthens the "flat representation is the limit" argument.

In [ ]:
results_dir = paths['regression_results'] / 'HUXt3' / 'phase2_architecture'

results_df = auto_detect_and_compare(
    results_dir=results_dir,
    save=True,
    save_dir=Path('analysis_outputs')
)

In [ ]:
from storm_regression.forecast_analysis import create_comparison_dashboard

# Usage example:
results_file = paths['regression_results'] / 'HUXt3' / 'phase2_architecture'
results_file = results_file / 'results_seed42_lt12_fold0_thresh4.5_balanced_Bz_GSM-Dst-n_sw_MLP_uniform_150_p5-16-50-84-95_arch150-150-150.pkl'

comparison_df = create_comparison_dashboard(
    results_path=results_file,
    prediction_type='y_pred_lognormal_median',
    save=True,
    save_name='dashboard_comparison.png'
)

In [ ]:
# Cell 3: Load and analyze specific result

results_file = paths['regression_results'] / 'HUXt3' / 'phase2_architecture'
results_file = results_file / 'results_seed42_lt12_fold0_thresh4.5_balanced_Bz_GSM-Dst-n_sw_MLP_uniform_150_p5-16-50-84-95_arch150-150-150.pkl'

case_study = analyze_results(
    filepath,
    n=100,
    event_types=['SIR'],
    descending=True,
    plot_function=plot_case_study,
)

# Phase 2 — Architecture (unbalanced check)

**Question:** does the architecture conclusion hold on the unbalanced (natural
class-ratio) test set, not just a balanced view?

**Finding:** [confirm from cells] the architecture ranking is unchanged under
the unbalanced evaluation — median-input parsimony still wins.

**Decision:** architecture choice robust to the storm/non-storm balance of the
evaluation set.

In [ ]:
# Usage example:
results_dir = paths['regression_results'] / 'HUXt3' / 'phase2_architecture_unbalanced_storms'

results_df = auto_detect_and_compare(
    results_dir=results_dir,
    test_mode='unbalanced',
    save=True,
    save_dir=Path('analysis_outputs')
)

In [ ]:
from storm_regression.forecast_analysis import create_comparison_dashboard

# Usage example:
results_file = paths['regression_results'] / 'HUXt3' / 'phase2_architecture_unbalanced_storms'
results_file = results_file / 'results_seed42_lt12_fold0_thresh4.5_unbalanced_Bz_GSM-Dst-n_sw_MLP_uniform_150_p5-16-50-84-95_arch150-150-150.pkl'

comparison_df = create_comparison_dashboard(
    results_path=results_file,
    prediction_type='y_pred_lognormal_median',
    save=True,
    save_name='dashboard_comparison.png'
)

In [ ]:
# Cell 3: Load and analyze specific result

results_file = paths['regression_results'] / 'HUXt3' / 'phase2_architecture_unbalanced_storms'

results = auto_detect_and_plot_case_studies(results_file)

# Phase 3 — Loss: NLL term coefficients

**Question:** do the NLL term weights (w_sigma, w_accuracy) materially change
the fit?

**Finding:** [confirm from cells] — sensitivity of skill/calibration to the NLL
coefficient balance.

**Decision:** [locked coefficients].

In [ ]:
## Phase 3 - Loss Functions

# Cell 3: Load and analyze specific result

results_file = paths['regression_results'] / 'HUXt3' / 'phase3_loss_functions_nll'
results_file = results_file / 'results_seed42_lt12_fold0_thresh4.5_unbalanced_Bz_GSM-Dst-n_sw_MLP_flat_p5-16-50-84-95_arch150-150-150_loss_typenll_w_sig1_w_acc0.5_hp30_weight1.0.pkl'

case_study = analyze_results(
    filepath,
    n=10,
    descending=True,
    plot_function=plot_case_study,
)

In [ ]:
results_file = paths['regression_results'] / 'HUXt3' / 'phase3_loss_functions_nll'
results_file = results_file / 'results_seed42_lt12_fold0_thresh4.5_unbalanced_Bz_GSM-Dst-n_sw_MLP_flat_p5-16-50-84-95_arch150-150-150_loss_typenll_w_sig0.5_w_acc1_hp30_weight1.0.pkl'

case_study = analyze_results(
    results_file,
    n=10,
    descending=True,
    plot_function=plot_case_study,
)

# Phase 3 — Loss: Hp30 (intensity) weighting

**Question:** does up-weighting storm windows in the loss fix the storm
undershoot — and does it improve skill, or just shift the bias?

**Finding:** weighting genuinely improves storm CRPS/PIT/bias — BUT only by
trading away quiet-period skill at equal-or-worse aggregate cost. The model
relocates accuracy along a storm-vs-quiet frontier; it cannot increase total
skill (it lacks the information to *discriminate* storms, only to shift its
global level).

**Decision:** final config stays UNWEIGHTED (best overall). Weighting is a
diagnostic, not a model choice — it reveals a discrimination ceiling.

In [ ]:
# Usage example:
results_dir = paths['regression_results'] / 'HUXt3' / 'phase3_loss_functions_hp30_weighting'

results_df = auto_detect_and_compare(
    results_dir=results_dir,
    test_mode='unbalanced',
    save=True,
    save_dir=Path('analysis_outputs')
)

In [ ]:
results_dir = paths['regression_results'] / 'HUXt3' / 'phase3_loss_functions_hp30_weighting'

results = auto_detect_and_plot_case_studies(results_dir)

# Phase 3 — Loss: distribution-parameter diagnostics
Visualising how the predicted median and sigma respond to storm strength under
each loss variant (checks whether sigma correctly rises on storms, whether the
median tracks or undershoots). Supports the weighting-frontier finding above.

In [ ]:
from storm_regression.forecast_analysis import plot_distribution_parameters

results_file = paths['regression_results'] / 'HUXt3' / 'phase3_loss_functions_nll'
results_file = results_file / 'results_seed42_lt12_fold0_thresh4.5_unbalanced_Bz_GSM-Dst-n_sw_MLP_flat_p5-16-50-84-95_arch150-150-150_loss_typenll_w_sig0.5_w_acc1_hp30_weight1.0.pkl'

distrubtions = plot_distribution_parameters(results_file)

In [ ]:
## Phase 3 - Loss Functions

# Cell 3: Load and analyze specific result

results_file = paths['regression_results'] / 'HUXt3' / 'phase3_loss_functions_nll'

case_study = auto_detect_and_plot_case_studies(
    results_file,
)

# Phase 3 — Loss function variety

**Question:** do alternative loss forms (asymmetric, CRPS-based) beat plain NLL?

**Finding:** [confirm from cells].

**Decision:** [locked loss].

In [ ]:
# Usage example:
results_dir = paths['regression_results'] / 'HUXt3' / 'phase3_loss_functions_hp30_weighting_scale_linear'

results_df = auto_detect_and_compare(
    results_dir=results_dir,
    test_mode='unbalanced',
    save=True,
    save_dir=Path('analysis_outputs')
)

In [ ]:
from storm_regression.forecast_analysis import plot_distribution_parameters
import logging
import re
results_dir = paths['regression_results'] / 'HUXt3' / 'phase3_loss_functions_hp30_weighting_scale_linear'

logging.disable(logging.INFO)

scales = [0.01, 0.1, 0.5, 1.0, 2.0, 5.0, 10.0, 50.0]

all_files = {float(re.search(r'hp30_scale([\d.]+)', f.stem).group(1)): f 
             for f in results_dir.glob('*.pkl')
             if re.search(r'hp30_scale([\d.]+)', f.stem)}

for scale in scales:
    if scale not in all_files:
        print(f"No file found for scale={scale}")
        continue
    plot_distribution_parameters(all_files[scale], logscale=False)

logging.disable(logging.NOTSET)

In [ ]:
## Phase 3 - Loss Functions

# Cell 3: Load and analyze specific result

results_file = paths['regression_results'] / 'HUXt3' / 'phase3_loss_functions_hp30_weighting_scale_linear'

case_study = auto_detect_and_plot_case_studies(
    results_file,
)

In [ ]:
# Usage example:
results_dir = paths['regression_results'] / 'HUXt3' / 'phase3_loss_functions_variety'

results_df = auto_detect_and_compare(
    results_dir=results_dir,
    test_mode='unbalanced',
    save=True,
    save_dir=Path('analysis_outputs')
)

In [ ]:
from storm_regression.forecast_analysis import plot_distribution_parameters
import logging

results_dir = paths['regression_results'] / 'HUXt3' / 'phase3_loss_functions_variety'

logging.disable(logging.INFO)

for filename in sorted(results_dir.glob('*.pkl')):
    print(f"\nPlotting: {filename.name}")
    plot_distribution_parameters(filename, logscale=False)

logging.disable(logging.NOTSET)

In [ ]:
## Phase 3 - Loss Functions

# Cell 3: Load and analyze specific result

results_file = paths['regression_results'] / 'HUXt3' / 'phase3_loss_functions_variety'

case_study = auto_detect_and_plot_case_studies(
    results_file,
)

In [ ]:
## Phase 3 - Loss Functions

# Cell 3: Load and analyze specific result

results_file = paths['regression_results'] / 'HUXt3' / 'phase3_loss_functions_variety'

case_study = auto_detect_and_plot_case_studies(
    results_file,
)

# Phase 3 — Ensemble selection & subsampling

**Question:** does *how* ensemble members are selected/subsampled (snap vs
per-timestep percentiles, member count) affect skill?

**Finding:** [confirm]. Given the median-input result, member-selection detail
is largely moot for the final flat-MLP config.

**Decision:** snap selection, n=200 retained for comparability.

In [ ]:
from storm_regression.forecast_analysis import plot_distribution_parameters
import logging

logging.disable(logging.INFO)

# Usage example:
results_dir = paths['regression_results'] / 'HUXt3' / 'phase3_ensemble_selection'

results_df = auto_detect_and_compare(
    results_dir=results_dir,
    test_mode='unbalanced',
    save=True,
    save_dir=Path('analysis_outputs')
)

# for filename in sorted(results_dir.glob('*.pkl')):
#     print(f"\nPlotting: {filename.name}")
#     plot_distribution_parameters(filename, logscale=False)

# case_study = auto_detect_and_plot_case_studies(
#     results_dir,
# )

logging.disable(logging.NOTSET)

In [ ]:
logging.disable(logging.INFO)

for filename in sorted(results_dir.glob('*.pkl')):
    print(f"\nPlotting: {filename.name}")
    plot_distribution_parameters(filename, logscale=False)

logging.disable(logging.NOTSET)

In [ ]:
case_study = auto_detect_and_plot_case_studies(
    results_dir,
)

In [ ]:
from storm_regression.forecast_analysis import plot_distribution_parameters
import logging

logging.disable(logging.INFO)

# Usage example:
results_dir = paths['regression_results'] / 'HUXt3' / 'phase3_ensemble_subsampling_with_raw_ensembles'

results_df = auto_detect_and_compare(
    results_dir=results_dir,
    test_mode='unbalanced',
    save=True,
    save_dir=Path('analysis_outputs')
)

# for filename in sorted(results_dir.glob('*.pkl')):
#     print(f"\nPlotting: {filename.name}")
#     plot_distribution_parameters(filename, logscale=False)

# case_study = auto_detect_and_plot_case_studies(
#     results_dir,
# )

logging.disable(logging.NOTSET)

In [ ]:
logging.disable(logging.INFO)

for filename in sorted(results_dir.glob('*.pkl')):
    print(f"\nPlotting: {filename.name}")
    plot_distribution_parameters(filename, logscale=False)

logging.disable(logging.NOTSET)

In [ ]:
case_study = auto_detect_and_plot_case_studies(
    results_dir,
)

In [ ]:
from storm_regression.forecast_analysis import plot_distribution_parameters
import logging

logging.disable(logging.INFO)

# Usage example:
results_dir = paths['regression_results'] / 'HUXt3' / 'phase3_ensemble_subsampling_per_timestep_nll'

results_df = auto_detect_and_compare(
    results_dir=results_dir,
    test_mode='unbalanced',
    save=True,
    save_dir=Path('analysis_outputs')
)

In [ ]:
for filename in sorted(results_dir.glob('*.pkl')):
    print(f"\nPlotting: {filename.name}")
    plot_distribution_parameters(filename, logscale=False)

In [ ]:
import logging

logging.disable(logging.INFO)

# Usage example:
results_dir = paths['regression_results'] / 'HUXt3' / 'phase3_ensemble_subsampling_per_timestep_nll'

case_study = auto_detect_and_plot_case_studies(
    results_dir,
    n_cases=10,
)

In [ ]:
from storm_regression.forecast_analysis import auto_detect_and_compare
import logging

logging.disable(logging.INFO)

# Usage example:
results_dir = paths['regression_results'] / 'HUXt3' / 'phase3_ensemble_subsampling_per_timestep_crps_loss'

results_df = auto_detect_and_compare(
    results_dir=results_dir,
    test_mode='unbalanced',
    save=True,
    save_dir=Path('analysis_outputs')
)

In [ ]:
from storm_regression.forecast_analysis import plot_distribution_parameters
results_dir = paths['regression_results'] / 'HUXt3' / 'phase3_ensemble_subsampling_per_timestep_crps_loss'

for filename in sorted(results_dir.glob('*.pkl')):
    print(f"\nPlotting: {filename.name}")
    plot_distribution_parameters(filename)

In [ ]:
import logging

logging.disable(logging.INFO)

# Usage example:
results_dir = paths['regression_results'] / 'HUXt3' / 'phase3_ensemble_subsampling_per_timestep_crps_loss'

case_study = auto_detect_and_plot_case_studies(
    results_dir,
    n_cases=20,
)

In [ ]:
from storm_regression.forecast_analysis import create_probabilistic_dashboard
from storm_utils.config_paths import get_project_paths

paths = get_project_paths()

results_dir = paths['regression_results'] / 'HUXt3' / 'phase3_ensemble_subsampling_per_timestep_crps_loss'
results_file = results_dir / 'results_keep100_snap_p5-16-50-84-95.pkl'

results = create_probabilistic_dashboard(results_file, family='auto')

# Phase 3 — Ensemble usefulness (ABC analysis)

**Question:** does the ensemble carry information beyond a point summary — an
information-content check independent of the MLP.

**Finding:** [confirm from run_abc_analysis output].

In [ ]:
from storm_regression.analyze_ensemble_usefulness import run_abc_analysis
from storm_utils.config_paths import get_project_paths
paths = get_project_paths()

import logging 
logging.disable(logging.INFO)

results_dir_A = paths['regression_results'] / 'HUXt3' / 'phase3_ensemble_subsampling_per_timestep_nll'
results_dir_B = paths['regression_results'] / 'HUXt3' / 'phase3_ensemble_subsampling_per_timestep_crps_loss'
results_dir_C = paths['regression_results'] / 'HUXt3' / 'phase3_ensemble_subsampling_with_raw_ensembles_nll'

groups = {
    "A per_ts+NLL":  results_dir_A,
    "B per_ts+CRPS": results_dir_B,
    "C snap+NLL":    results_dir_C,
}

df = run_abc_analysis(groups)

# Phase 3 - CRPS Loss
Implemented CRPS loss function

In [ ]:
import numpy as np
from storm_regression.results_io import load_results

results_dir = paths['regression_results'] / 'HUXt3' / 'phase3_ensemble_subsampling_per_timestep_crps_loss'
path = results_dir / 'results_keep100_snap_p5-16-50-84-95.pkl'

results, config, _ = load_results(path)
y   = np.asarray(results['y_test'])
mu  = np.asarray(results['log_mu']); sig = np.asarray(results['log_sigma'])
med = np.exp(mu); storm = y > 4.66

print(f"obs mean:     quiet={y[~storm].mean():.2f}   storm={y[storm].mean():.2f}")
print(f"pred median:  quiet={med[~storm].mean():.2f}   storm={med[storm].mean():.2f}")
print(f"pred sigma:   quiet={sig[~storm].mean():.3f}  storm={sig[storm].mean():.3f}")
print(f"corr(median,y) = {np.corrcoef(med, y)[0,1]:.3f}")
print(f"std(pred median)={med.std():.2f}  vs  std(obs)={y.std():.2f}")

# Phase 4 — Threshold-weighted CRPS loss

**Question:** does a tail-emphasising proper score (twCRPS) as the training
objective improve storm-tail skill over NLL?

**Finding:** [confirm from cells].

**Decision:** [outcome].

In [ ]:
# Probabilistic dashboard for regular CRPS

from storm_regression.forecast_analysis import create_probabilistic_dashboard
from storm_utils.config_paths import get_project_paths

paths = get_project_paths()

results_dir = paths['regression_results'] / 'HUXt3' / 'phase4_loss_comparison'
results_file = results_dir / 'results_loss_nll.pkl'

results = create_probabilistic_dashboard(results_file, family='auto')

In [ ]:
# Probabilistic dashboard for regular CRPS

from storm_regression.forecast_analysis import create_probabilistic_dashboard
from storm_utils.config_paths import get_project_paths

paths = get_project_paths()

results_dir = paths['regression_results'] / 'HUXt3' / 'phase4_loss_comparison'
results_file = results_dir / 'results_loss_crps.pkl'

results = create_probabilistic_dashboard(results_file, family='auto')

In [ ]:
# Probabilistic dashboard for twCRPS4.66

from storm_regression.forecast_analysis import create_probabilistic_dashboard
from storm_utils.config_paths import get_project_paths

paths = get_project_paths()

results_dir = paths['regression_results'] / 'HUXt3' / 'phase4_loss_comparison'
results_file = results_dir / 'results_loss_twcrps_t4.66.pkl'

results = create_probabilistic_dashboard(results_file, family='auto')

In [ ]:
from storm_regression.forecast_analysis import plot_distribution_parameters
results_dir = paths['regression_results'] / 'HUXt3' / 'phase4_loss_comparison'

for filename in sorted(results_dir.glob('*.pkl')):
    print(f"\nPlotting: {filename.name}")
    plot_distribution_parameters(filename)

In [ ]:
from storm_regression.forecast_analysis import auto_detect_and_compare
import logging
from pathlib import Path

logging.disable(logging.INFO)

# Usage example:
results_dir = paths['regression_results'] / 'HUXt3' / 'phase4_loss_comparison'

results_df = auto_detect_and_compare(
    results_dir=results_dir,
    test_mode='unbalanced',
    save=True,
    save_dir=Path('analysis_outputs')
)

# Phase 5a — Mixture density (LogNormal mixture)
**Question:** does a K-component mixture head capture storm-tail structure a
single LogNormal misses?

**Setup:** MLP trunk → K-component LogNormal mixture, NLL loss.

**Finding:** the mixture is a statistical TIE with single-LogNormal on storms
(CRPS/PIT/bias flat across K). No collapse — components are distinct and used —
but the extra components reshape the bulk/tails rather than forming a committed
storm mode. Storm PIT-KS remains ~0.55 regardless of K.

**Decision:** locked K=2 (matches the §3.1 thesis; K=3's marginal gain doesn't
justify the complexity). Mixture *form* is not the storm-tail bottleneck —
information is.

In [ ]:
from storm_regression.forecast_analysis import plot_distribution_parameters
results_dir = paths['regression_results'] / 'HUXt3' / 'phase5_mixture_nll'

for filename in sorted(results_dir.glob('*.pkl')):
    print(f"\nPlotting: {filename.name}")
    plot_distribution_parameters(filename)

In [ ]:
from storm_regression.forecast_analysis import auto_detect_and_compare
import logging
from pathlib import Path

logging.disable(logging.WARNING)

df = auto_detect_and_compare(
    results_dir=paths['regression_results'] / 'HUXt3' / 'phase5_mixture_nll',
    seed=42, fold=0,
)
# the 'model' column is now the distinct run_name — pull the all-data CRPS per run
print(df[df['subset'] == 'All test data'][['model', 'crps', 'rmse', 'mae']].to_string(index=False))

In [ ]:
# Probabilistic dashboard for mixture nll loss

from storm_regression.forecast_analysis import create_probabilistic_dashboard
from storm_utils.config_paths import get_project_paths

paths = get_project_paths()

results_dir = paths['regression_results'] / 'HUXt3' / 'phase5_mixture_nll'
results_file = results_dir / 'results_mixture_K2_nll_lr0p0001_pat20.pkl'

results = create_probabilistic_dashboard(results_file, family='auto')

In [ ]:
# Probabilistic dashboard for mixture nll loss

from storm_regression.forecast_analysis import create_probabilistic_dashboard
from storm_utils.config_paths import get_project_paths

paths = get_project_paths()

results_dir = paths['regression_results'] / 'HUXt3' / 'phase5_mixture_nll'
results_file = results_dir / 'results_mixture_K2_nll_lr0p001_pat20.pkl'

results = create_probabilistic_dashboard(results_file, family='auto')

In [ ]:
# Probabilistic dashboard for mixture nll loss

from storm_regression.forecast_analysis import create_probabilistic_dashboard
from storm_utils.config_paths import get_project_paths

paths = get_project_paths()

results_dir = paths['regression_results'] / 'HUXt3' / 'phase5_mixture_nll'
results_file = results_dir / 'results_mixture_K2_nll_lr0p0001_pat60.pkl'

results = create_probabilistic_dashboard(results_file, family='auto')

In [ ]:
# Probabilistic dashboard for mixture nll loss

from storm_regression.forecast_analysis import create_probabilistic_dashboard
from storm_utils.config_paths import get_project_paths

paths = get_project_paths()

results_dir = paths['regression_results'] / 'HUXt3' / 'phase5_mixture_nll'
results_file = results_dir / 'results_mixture_K2_nll_lr0p001_pat60.pkl'

results = create_probabilistic_dashboard(results_file, family='auto')

In [ ]:
results_dir = paths['regression_results'] / 'HUXt3' / 'phase5_mixture_nll'

case_study = auto_detect_and_plot_case_studies(
    results_dir,
)

In [ ]:
import numpy as np
from storm_regression.results_io import load_results
from storm_regression.predictive import forecast_from_results, LogNormalForecast

results_dir = paths['regression_results'] / 'HUXt3' / 'phase5_mixture_nll'
results_file = results_dir / 'results_mixture_K2_nll.pkl'
results, config, _ = load_results(results_file)

# (a) confirm what got scored
f = forecast_from_results(results, family='auto')
print("forecast type:", type(f).__name__)          # must be MixtureForecast

# (b) sane medians and exceedance?
print("median range:", f.median().min(), f.median().max())   # expect ~2-7
print("median sample:", f.median()[:5])
print("P(>4.66) range:", f.exceedance_prob(4.66).min(), f.exceedance_prob(4.66).max())

# (c) compare the mixture's CRPS to the top-component-only LogNormal on the SAME data
y = np.asarray(results['y_test'])
crps_mix = f.crps(y).mean()
f_single = LogNormalForecast(results['log_mu'], results['log_sigma'])  # the fallback component
crps_single = f_single.crps(y).mean()
print(f"mixture CRPS={crps_mix:.3f}   single-(top-comp) CRPS={crps_single:.3f}")

# (d) what did training actually reach?
# (look at the logged 'Best loss' for this run vs your single-LogNormal run)

In [ ]:
## Check if validation loss stopped the overfitting

from storm_regression.results_io import load_results
from storm_regression.predictive import forecast_from_results
import numpy as np

results_file1 = paths['regression_results'] / 'HUXt3' / 'phase5_mixture_nll_validation_early_stopping' / 'results_mixture_K2_nll_lr0p001_pat60_valstop.pkl'
results_file2 = paths['regression_results'] / 'HUXt3' / 'phase5_mixture_nll' / 'results_mixture_K2_nll_lr0p001_pat60.pkl'

for results_file in [results_file1, results_file2]:
    results, config, _ = load_results(results_file)
    f = forecast_from_results(results, family='auto')
    print("test CRPS:", f.crps(np.asarray(results['y_test'])).mean())

In [ ]:
## Check if loss curve plotting works

from storm_regression.plotting import plot_loss_curve

results_file = paths['regression_results'] / 'HUXt3' / 'phase5_mixture_nll_validation_early_stopping' / 'results_mixture_K2_nll_lr0p001_pat60_valstop.pkl'
plot_loss_curve(results_file)

In [ ]:
# Check case studies with validation loss

'''
verify:
    1. Are the case studies still the same as what we saw previously? 
    2. What are the shapes of the distributions?
'''

results_dir = paths['regression_results'] / 'HUXt3' / 'phase5_mixture_nll_validation_early_stopping'
results = auto_detect_and_plot_case_studies(results_dir)

In [ ]:
# Check case studies for varying percentiles

'''
verify:
    1. What are the shapes of the distributions for varying percentiles used?
'''

results_dir = paths['regression_results'] / 'HUXt3' / 'phase5_mixture_nll_percentiles_search'
results = auto_detect_and_plot_case_studies(results_dir)

In [ ]:
results_dir = paths['regression_results'] / 'HUXt3' / 'phase5_mixture_nll_percentiles_search'
auto_detect_and_compare(results_dir)

In [ ]:
results_dir = paths['regression_results'] / 'HUXt3' / 'phase5_mixture_nll_percentiles_search'
results_file  = results_dir / 'results_mixture_K2_nll_perc_p50_valstop.pkl'

results = create_probabilistic_dashboard(results_file)

## Collapse diagnostic
Checks whether the mixture is degenerate (a component collapsing to ~0 weight or
components sharing a location). Rules out mode collapse as the reason multi­
modality is mild — confirms the components are genuinely distinct and used.

In [ ]:
import numpy as np
from storm_regression.results_io import load_results

results_dir = paths['regression_results'] / 'HUXt3' / 'phase5_mixture_nll_percentiles_search'
results_file  = results_dir / 'results_mixture_K2_nll_perc_p50_valstop.pkl'
results, config, _ = load_results(results_file)

mu  = np.asarray(results['mu_m'])      # (B, K)
sig = np.asarray(results['sigma_m'])   # (B, K)
w   = np.asarray(results['alpha'])     # (B, K)

# 1. Are the two components' means actually separated, per window?
mu_gap = np.abs(mu[:, 0] - mu[:, 1])
print(f"|mu_0 - mu_1|:   mean={mu_gap.mean():.3f}  median={np.median(mu_gap):.3f}  max={mu_gap.max():.3f}")

# 2. Are both components actually used? (weights not pinned to one)
print(f"weight on comp 0: mean={w[:,0].mean():.3f}  min={w[:,0].min():.3f}  max={w[:,0].max():.3f}")
frac_decisive = np.mean((w.max(axis=1) > 0.95))
print(f"fraction of windows >95% one component: {frac_decisive:.2%}")

# 3. Do the components separate MORE on storm windows? (the thing we WANT)
storm = results['y_test'] > 4.66
print(f"mean |mu gap| quiet={mu_gap[~storm].mean():.3f}  storm={mu_gap[storm].mean():.3f}")

# 4. Verdict
if mu_gap.mean() < 0.05 and w[:,0].std() < 0.05:
    print(">>> LIKELY COLLAPSED: components nearly identical and weights barely vary.")
else:
    print(">>> Components distinct and/or weights adapt per window — not collapsed.")

In [ ]:
import numpy as np
from storm_regression.results_io import load_results

results_dir = paths['regression_results'] / 'HUXt3' / 'phase5_mixture_nll_percentiles_search'
results_file  = results_dir / 'results_mixture_K2_nll_perc_p5s_valstop.pkl'
results, config, _ = load_results(results_file)

mu  = np.asarray(results['mu_m'])      # (B, K)
sig = np.asarray(results['sigma_m'])   # (B, K)
w   = np.asarray(results['alpha'])     # (B, K)

# 1. Are the two components' means actually separated, per window?
mu_gap = np.abs(mu[:, 0] - mu[:, 1])
print(f"|mu_0 - mu_1|:   mean={mu_gap.mean():.3f}  median={np.median(mu_gap):.3f}  max={mu_gap.max():.3f}")

# 2. Are both components actually used? (weights not pinned to one)
print(f"weight on comp 0: mean={w[:,0].mean():.3f}  min={w[:,0].min():.3f}  max={w[:,0].max():.3f}")
frac_decisive = np.mean((w.max(axis=1) > 0.95))
print(f"fraction of windows >95% one component: {frac_decisive:.2%}")

# 3. Do the components separate MORE on storm windows? (the thing we WANT)
storm = results['y_test'] > 4.66
print(f"mean |mu gap| quiet={mu_gap[~storm].mean():.3f}  storm={mu_gap[storm].mean():.3f}")

# 4. Verdict
if mu_gap.mean() < 0.05 and w[:,0].std() < 0.05:
    print(">>> LIKELY COLLAPSED: components nearly identical and weights barely vary.")
else:
    print(">>> Components distinct and/or weights adapt per window — not collapsed.")

In [ ]:
import numpy as np
from storm_regression.results_io import load_results

results_dir = paths['regression_results'] / 'HUXt3' / 'phase5_mixture_nll_percentiles_search'
results_file  = results_dir / 'results_mixture_K2_nll_perc_p1s_valstop.pkl'
results, config, _ = load_results(results_file)

mu  = np.asarray(results['mu_m'])      # (B, K)
sig = np.asarray(results['sigma_m'])   # (B, K)
w   = np.asarray(results['alpha'])     # (B, K)

# 1. Are the two components' means actually separated, per window?
mu_gap = np.abs(mu[:, 0] - mu[:, 1])
print(f"|mu_0 - mu_1|:   mean={mu_gap.mean():.3f}  median={np.median(mu_gap):.3f}  max={mu_gap.max():.3f}")

# 2. Are both components actually used? (weights not pinned to one)
print(f"weight on comp 0: mean={w[:,0].mean():.3f}  min={w[:,0].min():.3f}  max={w[:,0].max():.3f}")
frac_decisive = np.mean((w.max(axis=1) > 0.95))
print(f"fraction of windows >95% one component: {frac_decisive:.2%}")

# 3. Do the components separate MORE on storm windows? (the thing we WANT)
storm = results['y_test'] > 4.66
print(f"mean |mu gap| quiet={mu_gap[~storm].mean():.3f}  storm={mu_gap[storm].mean():.3f}")

# 4. Verdict
if mu_gap.mean() < 0.05 and w[:,0].std() < 0.05:
    print(">>> LIKELY COLLAPSED: components nearly identical and weights barely vary.")
else:
    print(">>> Components distinct and/or weights adapt per window — not collapsed.")

## K-sweep (K=1 / K=2 / K=3 / K=5)
**Question:** does adding mixture components improve storm calibration?

**Finding:** storm CRPS/PIT-KS/bias essentially flat K1→K3; K3 marginally best
on aggregate calibration only. Confirms mixture order is not the storm-tail
limit.

**Decision:** K=2.

In [ ]:
import numpy as np
from storm_regression.results_io import load_results

results_dir = paths['regression_results'] / 'HUXt3' / 'phase5_mixture_nll_K_sweep'

In [ ]:
from storm_regression.forecast_analysis import create_probabilistic_dashboard

results_file_K1  = results_dir / 'results_mixture_K1_nll_valstop.pkl'
c1 = create_probabilistic_dashboard(results_file_K1)

results_file_K2  = results_dir / 'results_mixture_K2_nll_valstop.pkl'
c2 = create_probabilistic_dashboard(results_file_K2)

results_file_K3  = results_dir / 'results_mixture_K3_nll_valstop.pkl'
c3 = create_probabilistic_dashboard(results_file_K3)

results_file_K5  = results_dir / 'results_mixture_K5_nll_valstop.pkl'
c5 = create_probabilistic_dashboard(results_file_K5)

In [ ]:
auto_detect_and_compare(results_dir)

In [ ]:
## Phase 3a.5 - K sweep comparison table
import numpy as np
from scipy.stats import kstest
from storm_regression.results_io import load_results, recreate_dataset_from_results
from storm_regression.predictive import forecast_from_results

sweep_dir = paths['regression_results'] / 'HUXt3' / 'phase5_mixture_nll_K_sweep'
K_values = [1, 2, 3, 5]
storm_threshold = 4.66

def effective_components(results, weight_floor=0.05):
    """Mean number of components carrying > weight_floor weight per window.
    For a single LogNormal (no alpha) this is 1 by definition."""
    if 'alpha' not in results:
        return 1.0
    w = np.asarray(results['alpha'])                  # (B, K)
    return float((w > weight_floor).sum(axis=1).mean())

def pit_ks(forecast, y, idx=None):
    """KS statistic of the PIT values (0 = perfectly calibrated/uniform)."""
    f = forecast.subset(idx) if idx is not None else forecast
    yy = y[idx] if idx is not None else y
    pit = f.pit(yy)
    return float(kstest(np.clip(pit, 0, 1), 'uniform').statistic)

rows = []
for K in K_values:
    fp = sweep_dir / f"results_mixture_K{K}_nll_valstop.pkl"
    results, config, _ = load_results(fp)
    dataset = recreate_dataset_from_results(fp)

    y = np.asarray(results['y_test'], dtype=float)
    f = forecast_from_results(results, family='auto')

    test_wps = config['test_indices']
    pos2idx = {wp: i for i, wp in enumerate(test_wps)}
    storm_wps = dataset.filter_indices_by_storm_strength(
        list(test_wps), min_strength=storm_threshold, max_strength=None)
    storm_idx = np.array([pos2idx[wp] for wp in storm_wps if wp in pos2idx], dtype=int)

    # subset the FORECAST to the storm windows so shapes match the storm observations
    if len(storm_idx):
        f_storm = f.subset(storm_idx)
        crps_storm = float(f_storm.crps(y[storm_idx]).mean())
    else:
        crps_storm = np.nan

    rows.append({
        'K': K,
        'test_CRPS': float(f.crps(y).mean()),
        'CRPS_storm': crps_storm,
        'PITKS_all': pit_ks(f, y),
        'PITKS_storm': pit_ks(f, y, storm_idx) if len(storm_idx) else np.nan,
        'eff_comp': effective_components(results),
        'n_storm': len(storm_idx),
    })

# ---- print table ----
print(f"\n{'K':>3}{'test CRPS':>11}{'CRPS storm':>12}{'PITKS all':>11}"
      f"{'PITKS storm':>13}{'eff comp':>10}{'n storm':>9}")
print("-" * 69)
for r in rows:
    print(f"{r['K']:>3}{r['test_CRPS']:>11.3f}{r['CRPS_storm']:>12.3f}"
          f"{r['PITKS_all']:>11.3f}{r['PITKS_storm']:>13.3f}"
          f"{r['eff_comp']:>10.2f}{r['n_storm']:>9}")

In [ ]:
import numpy as np
from scipy.stats import kstest
from storm_regression.results_io import load_results, recreate_dataset_from_results
from storm_regression.predictive import forecast_from_results

gate_dir = paths['regression_results'] / 'HUXt3' / 'phase5b_gate_diagnostic'
st = 4.66
reps = ['median_only', 'full_perc', 'p5s', 'p1s']
lead = 12

print(f"{'arm':>14}{'CRPS storm':>12}{'pred(S)':>9}{'obs(S)':>8}{'gap':>7}{'PITKS storm':>13}")
print("-" * 64)
for label in reps:
    fp = gate_dir / f"results_gate_{label}_lt{lead}.pkl"
    r, cfg, _ = load_results(fp)
    ds = recreate_dataset_from_results(fp)
    y = np.asarray(r['y_test'], float)
    f = forecast_from_results(r, family='auto')
    p2i = {wp: i for i, wp in enumerate(cfg['test_indices'])}
    sw = ds.filter_indices_by_storm_strength(list(cfg['test_indices']), min_strength=st, max_strength=None)
    s = np.array([p2i[w] for w in sw if w in p2i], int)
    med = f.median()
    crps_s = float(f.subset(s).crps(y[s]).mean())
    pit_s = float(kstest(np.clip(f.subset(s).pit(y[s]), 0, 1), 'uniform').statistic)
    predS, obsS = float(med[s].mean()), float(y[s].mean())
    print(f"{label:>14}{crps_s:>12.3f}{predS:>9.2f}{obsS:>8.2f}{obsS-predS:>7.2f}{pit_s:>13.3f}")

In [ ]:
auto_detect_and_compare(gate_dir)

In [ ]:
import numpy as np
from collections import Counter
from storm_utils.data_structure import ForecastingDataset
from storm_utils.config_paths import get_project_paths

huxt_run_id = 3
huxt_path = paths['huxt_data_shared'] / f'HUXt{huxt_run_id}_modified' / 'full_df.parquet'
discontinuity_path = paths['huxt_data_shared'] / f'HUXt{huxt_run_id}_modified' / 'discontinuities.npy'

n_ensembles = 1
Lt = 6
F = 24
stride_hours = 24

ds = ForecastingDataset(huxt_path, discontinuity_path, Nens=n_ensembles, lead_time_hours=Lt, forecast_duration_hours=F, stride_hours=stride_hours)

threshold = 4.5  # match ForecastingConfig.DEFAULT_STORM_THRESHOLD (see note below)

max_targets = np.asarray(ds.max_targets)
labels = np.asarray(ds.window_labels)

storm_mask = max_targets >= threshold
cme_mask = np.isin(labels, ['ICME_input', 'ICME_forecast'])

n_total          = len(max_targets)
n_storm          = storm_mask.sum()
n_cme            = cme_mask.sum()
n_storm_and_cme  = (storm_mask & cme_mask).sum()      # storms removed BECAUSE of CME
n_storm_kept     = (storm_mask & ~cme_mask).sum()     # storms surviving CME removal

print(f"Total windows:            {n_total}")
print(f"Storms (>= {threshold}):        {n_storm}")
print(f"CME windows:              {n_cme}")
print(f"Storm AND CME:            {n_storm_and_cme}")
print(f"Storms surviving:         {n_storm_kept}")
print(f"Fraction of storms that are CME: {n_storm_and_cme / n_storm * 100:.1f}%")

In [ ]:
## New CR Split test
from storm_regression.forecast_analysis import create_probabilistic_dashboard

results_dir = paths['regression_results'] / 'HUXt3' / 'phase5b_new_CR_split_test' / 'results_gate_p5s_lt12.pkl'
results_df = create_probabilistic_dashboard(results_dir)

## Final flat-MLP validation (control)

**Question:** locked-config skill with proper error bars, as the baseline the
set encoder must beat.

**Setup:** K=2, deep_narrow, median, unweighted, 5 seeds × 5 folds × {6,12,24}h.

**Finding:** storm CRPS ~1.0/1.06/1.14 (6/12/24h); storm PIT-KS 0.52/0.55/0.60
(catastrophic, worsens with lead); persistent storm undershoot (bias −1.4 to
−1.65); all-test well-calibrated (PIT-KS ~0.04). Smooth monotonic degradation.

**Decision:** this is the reported §5.1 control baseline. Numbers to beat with
the encoder: storm CRPS ~1.06, storm PIT-KS ~0.55 at 12h.

In [ ]:
from storm_regression.forecast_analysis import auto_detect_and_compare
from storm_utils.config_paths import get_project_paths

paths = get_project_paths()
results_dir = paths['regression_results'] / 'HUXt3' / 'phase5a_features'

df = auto_detect_and_compare(
    results_dir=results_dir,
    seed=42, fold=0, threshold=4.5, test_mode='unbalanced',
    include_baselines=True,
)

In [ ]:
# View the storm time subset
df[df['subset'] == 'Storms (>4.5)']

In [ ]:
results_dir = paths['regression_results'] / 'HUXt3' / 'phase5a_features_allfolds'

df_allfolds = auto_detect_and_compare(results_dir)

In [ ]:
#if auto_detect_and_compare is funny...

import pandas as pd
dfs = []
for f in range(5):
    d = auto_detect_and_compare(
        results_dir=paths['...'] / 'phase1_features_allfolds',
        seed=42, fold=f, threshold=4.5, test_mode='unbalanced',
        include_baselines=True,
    )
    d['fold'] = f
    dfs.append(d)
allf = pd.concat(dfs)
summary = allf.groupby(['model','subset']).agg(
    crps_mean=('crps','mean'), crps_std=('crps','std'),
    rmse_mean=('rmse','mean'), rmse_std=('rmse','std'),
).reset_index()

In [ ]:
import pickle
from pathlib import Path
import numpy as np
import pandas as pd
from storm_regression.predictive import forecast_from_results
import re

def quick_metrics(results_dir, threshold=4.5):
    rows = []
    for f in sorted(Path(results_dir).glob('*.pkl')):
        with open(f, 'rb') as fh:
            pkg = pickle.load(fh)
        r, cfg = pkg['results'], pkg['config']
        fold = cfg['test_fold']
        run_label = re.sub(r'_seed\d+_lt\d+_fold\d+$', '', f.stem).replace('results_', '', 1)  

        y = np.asarray(r['y_test']).ravel()
        # point prediction key — check yours (lognormal_median for K=1)
        yhat = np.asarray(r['y_pred_lognormal_median']).ravel()

        # recompute CRPS from stored dist params via the VERIFIED path
        fc = forecast_from_results(r, family='auto')
        crps = np.asarray(fc.crps(y)).ravel()

        for subset, mask in [
            ('all',   np.ones_like(y, bool)),
            ('storm', y > threshold),
        ]:
            if mask.sum() == 0:
                continue
            yy, hh, cc = y[mask], yhat[mask], crps[mask]
            rows.append({
                'run':   run_label,
                'file':  f.stem,
                'fold':  cfg['test_fold'],
                'subset': subset,
                'n':     int(mask.sum()),
                'rmse':  float(np.sqrt(np.mean((hh - yy)**2))),
                'mae':   float(np.mean(np.abs(hh - yy))),
                'crps':  float(np.mean(cc)),
            })
    return pd.DataFrame(rows)

results_dir = paths['regression_results'] / 'HUXt3' / 'phase5a_features_allfolds'
df = quick_metrics(results_dir)

# aggregate across folds — the mean+std you actually want:
summary = (df.groupby(['run', 'subset'])
             .agg(crps_mean=('crps','mean'), crps_std=('crps','std'),
                  rmse_mean=('rmse','mean'), rmse_std=('rmse','std'),
                  n_folds=('fold','nunique'))
             .reset_index())

In [ ]:
## All events
summary[summary['subset'] == 'all']

In [ ]:
## Lowest scoring CRPS runs (note all fold 4)
df[df['crps'] < 0.62]

In [ ]:
### Architecture check

In [ ]:
results_dir = paths['regression_results'] / 'HUXt3' / 'phase5a_architecture'
auto_detect_and_compare(results_dir)

In [ ]:
from storm_regression.forecast_analysis import fast_batch_metrics

results_dir = paths['regression_results'] / 'HUXt3' / 'phase5a_architecture'
fast_batch_metrics(results_dir)

In [ ]:
from storm_regression.forecast_analysis import fast_batch_metrics

results_dir = paths['regression_results'] / 'HUXt3' / 'phase5a_architecture_foldvalidation'
df = fast_batch_metrics(results_dir)
agg = aggregate_folds(df)

# df[df['subset'] == 'all']
agg

In [ ]:
### Quick K-Sweep Metrics

from storm_regression.forecast_analysis import fast_batch_metrics, aggregate_folds

results_dir = paths['regression_results'] / 'HUXt3' / 'phase5a_ksweep'
df = fast_batch_metrics(results_dir)
agg = aggregate_folds(df)

# df[df['subset'] == 'all']
agg

In [ ]:
### Quick Intensity Weighting check

from storm_regression.forecast_analysis import fast_batch_metrics, aggregate_folds

results_dir = paths['regression_results'] / 'HUXt3' / 'phase5a_mixture_intensity_weighting'
df = fast_batch_metrics(results_dir)
agg = aggregate_folds(df)

# df[df['subset'] == 'all']
df

In [ ]:
results_dir = paths['regression_results'] / 'HUXt3' / 'phase5a_mixture_intensity_weighting'
auto_detect_and_compare(results_dir)

In [ ]:
### Quick Intensity Weighting check

from storm_regression.forecast_analysis import fast_batch_metrics, aggregate_folds

results_dir = paths['regression_results'] / 'HUXt3' / 'phase5a_final_control'
df = fast_batch_metrics(results_dir)
agg = aggregate_folds(df)

# df[df['subset'] == 'all']
df

In [ ]:
agg

In [ ]:
# Components of the distribution: 
from storm_regression.plotting import plot_mixture_single, plot_mixture_directory

results_dir = paths['regression_results'] / 'HUXt3' / 'phase5a_final_control'

for lt in [6, 12, 24]:
    results_file = results_dir / f'results_control_k2_deepnarrow_median_seed100_lt{lt}_fold0.pkl'

    plot_mixture_single(results_file)

# Phase 5b — Set encoder (Deep Sets)
**Question:** does permutation-invariant, member-level processing extract
storm-discriminating information the median throws away — i.e. is the ceiling
representational or informational?

## Discovery (mean-pool, K=2, v-only, 12h, fold 0)
**Finding:** mean-pooled Deep Sets TIES the control (storm CRPS ~1.01, bias
~−1.52 — both within noise of the flat MLP). Mean-pooling ≈ member central
tendency ≈ the median, so a tie is expected; this is the weak-baseline control
within the encoder family, not the real test. Real test is attention (learned
per-member weighting = §5.3).

In [ ]:
results_dir = paths['regression_results'] / 'HUXt3' / 'phase5b_setencoder'
df = auto_detect_and_compare(results_dir)

In [ ]:
df

## K=3 Mixture Behaviour — Interpretation (mean-pool set encoder, unweighted loss)

**Run:** mean-pooled set encoder, K=3 LogNormal mixture, v-only, 12h lead,
`loss.intensity_type = 'none'` (no storm-time weighting).

### Observed structure

The three components do **not** contribute equally. The learned mixture
decomposes into:

- **One dominant (bulk) component** — carries most of the weight (highest α)
  and accounts for the central mass of the predictive distribution. Its
  location (mu) shifts *slightly* between storm and non-storm windows, but the
  shift is modest — the component is not strongly committal to either regime.
- **Two lower-weight flanking components** (lower α) that sit either side of the
  bulk in location:
  - one with **mu higher** than the bulk (upper flank),
  - one with **mu lower** than the bulk (lower flank).
  These appear to act mainly to **reshape the tails** of the predictive
  distribution rather than to represent a distinct storm-time regime.

### Reading

- There **is** genuine multimodality present — the components are distinct and
  used, not collapsed — but it is **mild**: the flanking components adjust the
  shape/skew of the distribution rather than forming a well-separated,
  high-weight "storm mode".
- The bulk component's weak storm/quiet shift, combined with the flanks doing
  tail-shaping rather than committing, suggests the model is representing a
  broadly **unimodal-but-skewed conditional distribution**, flexibly fitted,
  rather than a sharply bimodal storm-vs-quiet outcome.

### Caveats / likely causes

- **Unweighted loss.** With `intensity_type='none'`, the objective is dominated
  by the (majority) non-storm windows, so there is no pressure for a component
  to *commit* to storms. The mild, tail-shaping use of the flanking components
  is consistent with this — storm-time weighting might push a component to
  commit, but (per the intensity-weighting diagnostic) likely at the cost of the
  bulk rather than by gaining discrimination.
- **Revised expectation.** The

In [ ]:
## Three component mixture (K=3)
from storm_utils.config_paths import get_project_paths

paths = get_project_paths()

results_dir = paths['regression_results'] / 'HUXt3' / 'phase5b_setencoder_3components'

df = fast_batch_metrics(results_dir)
df

In [ ]:
from storm_regression.plotting import plot_mixture_single, plot_mixture_directory

results_dir = paths['regression_results'] / 'HUXt3' / 'phase5b_setencoder_3components'
results_file = results_dir / 'results_p5b_mean_k3_vonly_seed42_lt12_fold0.pkl'

plot_mixture_single(results_file)

## K=2 Mixture Behaviour — Interpretation (mean-pool set encoder, unweighted loss)

**Run:** mean-pooled set encoder, K=2 LogNormal mixture, v-only per-member
(OMNI + historic Hp30 as pooled context), 12h lead, `loss.intensity_type='none'`.

### Observed structure

The two components **almost completely separate in weight**: one component
dominates with ~90% of the mixture weight across most forecasts. The dominant
component carries the bulk of the predictive distribution; the minor component
(~10% weight) contributes only a subtle shape adjustment at low Hp30 values —
slightly pulling the distribution down during non-storm windows — rather than
representing a distinct high-intensity (storm) mode.

The dominant (bulk) component's weight is **even higher during storm windows**,
so the mixture becomes *more* single-component, not less, exactly where a second
mode would be most useful.

### Reading

- This is effectively a **weight collapse, not a location collapse**. The two
  components do *not* converge to the same mu (their locations stay distinct) —
  but because the minor component is held at ~10% weight (and lower on storms),
  the mixture behaves as a single dominant mode with a minor low-side shape
  correction. It "collapses in alpha, but not in mu."
- Functionally, K=2 here reduces to a single-component fit plus a small,
  non-committal tweak to the low-Hp30 shape. There is **essentially no working
  multimodality** — the second component is not being used to represent a storm
  regime, only to nudge the bulk's low tail during quiet periods.

### Comparison to K=3

Contrasts with the K=3 case, where the extra components produce a visibly richer
(if still mild) structure — one bulk plus two flanking components either side of
it. K=2 does not reproduce that: with only one spare component, the model spends
it on a low-side bulk correction rather than a high-side storm mode, and
suppresses it further on storms. This motivates preferring K=3 for interpretable
component structure — noting (from the earlier K-sweep) that K does **not**
change held-out skill (CRPS/PIT are K-tied), so this is an interpretability
choice, not a skill improvement, and implies updating the §3.1 two-component
framing if K=3 is adopted.

### Caveat / likely cause

Consistent with the unweighted loss (`intensity_type='none'`): with no storm-time
penalty, the objective is dominated by non-storm windows, so there is no pressure
to reserve a component for the storm tail — the model rationally concentrates
weight on the bulk and uses the spare component for a minor low-side correction.
The increased bulk dominance *on storms* is the same discrimination-limit
signature seen throughout: lacking storm-distinguishing information, the model
does not commit a mode to storms even when one is available to it.

In [ ]:
# Two component mixture
from storm_utils.config_paths import get_project_paths

paths = get_project_paths()

results_dir = paths['regression_results'] / 'HUXt3' / 'phase5b_setencoder'

df = fast_batch_metrics(results_dir)
df

In [ ]:
from storm_regression.plotting import plot_mixture_single, plot_mixture_directory

results_dir = paths['regression_results'] / 'HUXt3' / 'phase5b_setencoder'
results_file = results_dir / 'results_p5b_meanpool_k2_vonly_seed42_lt12_fold0.pkl'

plot_mixture_single(results_file)
# plot_mixture_directory(results_dir)

## K-Sweep | K $\in$ {1, 2, 3, 4, 5, 10}

**Question:** how does the number of mixture components affect (a) held-out skill
and (b) the learned component structure, for the mean-pool set encoder?

**Setup:** mean-pooled set encoder, v-only, 12h, fold 0, seed 42,
`intensity_type='none'`. `n_components` swept over {1, 2, 3, 4, 5, 10}; all other
settings fixed. (Note: `n_components` must be set on the loss config attached to
each run's `TrainingConfig` — the run_name label alone does not control K; K is
verified from `mu_m.shape[1]` in the saved results.)

**Finding — skill:** [fill from fast_batch_metrics/aggregate_folds] — storm and
all-test CRPS/PIT-KS across K. Prior K-sweeps found skill essentially flat across
K (mixture order is not the storm-tail bottleneck); confirm whether that holds
here for the set encoder.

**Finding — structure:** [fill from plot_mixture_single per K] — how many
components are genuinely used (non-negligible alpha) vs near-dead at each K, and
whether higher K produces more distinct modes or just more redundant components
clustered on the bulk. Cross-reference the hot-start result: structure at a given
K may depend heavily on initialisation, so distinguish "K allows more structure"
from "K plus a spread init allows more structure".

**Decision:** [K choice + rationale]. If skill is K-tied, the choice rests on
interpretable structure + thesis fit; note that adopting K>2 requires updating
the §3.1 two-component framing.

In [ ]:
# K sweep on the mixtures
from storm_utils.config_paths import get_project_paths

paths = get_project_paths()

results_dir = paths['regression_results'] / 'HUXt3' / 'phase5b_setencoder_ksweep'

df = fast_batch_metrics(results_dir)
df

In [ ]:
# Components of the distribution: 
from storm_regression.plotting import plot_mixture_single, plot_mixture_directory

results_dir = paths['regression_results'] / 'HUXt3' / 'phase5b_setencoder_ksweep'

for k in [2, 3, 4, 5, 10]:
    print(f'=== K = {k} ===')
    results_file = results_dir / f'results_p5b_k{k}_vonly_seed42_lt12_fold0.pkl'
    plot_mixture_single(results_file)

## Hot-Start Initialisation Test (K=3 mean-pool, unweighted loss)

**Run:** mean-pooled set encoder, K=3, v-only per-member (OMNI + historic Hp30
as pooled context), 12h lead, seed 42, fold 0, `loss.intensity_type='none'`.
A/B: `se_init_spread=False` (default) vs `True` (component medians spread across
Hp30 [1, 7] at initialisation; symmetry broken in mu location only, sigma and
alpha left symmetric).

### Observed behaviour (hot-start ON)

Three genuinely distinct, all-used components emerge — a marked improvement on
the default-init runs:

- **Component 1 (bulk anchor):** dominant weight (alpha ~0.5–0.95, median ~0.76),
  carries the central mass. Still the primary representation.
- **Component 0 (lower non-storm):** alpha ~0.05–0.5 (median ~0.2), sits at lower
  Hp30 than the bulk — a genuine second low-side mode, now carrying real weight
  (unlike the default runs where the minor component was near-dead).
- **Component 2 (tail / occasional storm):** alpha ~0.03–0.12, median centred
  ~4.5 Hp30 — sometimes a further non-storm component, sometimes sitting in the
  storm range. This is the closest any run has come to a dedicated high-Hp30
  component, though its weight remains low.

The component-0-vs-1 median scatter shows occasional collapse to a single mode,
but the components stay separated for the majority of forecasts.

### Key finding

Hot-starting **materially changes the learned structure**. Under the *same*
data, loss, and architecture, the default (clustered) initialisation for this
seed collapsed completely to effectively one component, whereas the spread
initialisation produced three distinct, all-used components with a genuine
(if low-weight) high-Hp30 mode. The high component was *given* a starting
location in the tail and — crucially — **did not fully migrate back to the bulk
or die**: it retained a distinct location (~4.5) and non-trivial weight after
training.

### Interpretation — "failure to separate", not "collapse"

This reframes the earlier collapse observations. The mixture components may not
have been *collapsing* (being actively pulled together by the data); rather, they
were **initialised too close together and never separated** — under a bulk-
dominated unweighted loss there is little gradient pressure to push a component
out to the tail from a clustered start. The hot-start provides that separation
for free, and the resulting structure then *survives* training, which indicates
the multimodal structure is *supportable* by the data — it simply wasn't being
*discovered* from a neutral start.

This is an important distinction:
- **Collapse (data-driven):** components pulled together because the conditional
  distribution is genuinely unimodal → hot-start would fail (components migrate
  back). NOT what we see.
- **Failure to separate (optimisation-driven):** components never explore the
  tail from a clustered init → hot-start fixes it, structure survives. This IS
  what we see.

So the mild/absent multimodality in prior runs was at least partly an
**initialisation/optimisation artefact**, not purely an information ceiling.
Caveat: the high (storm) component's weight remains low under the unweighted
loss — location is recoverable via init, but committing *weight* to it may still
require a storm-emphasising loss. Single seed/fold — the run-to-run consistency
of this behaviour is the subject of the follow-up hot-start exploration.

In [ ]:
# Components of the distribution: 
from storm_regression.plotting import plot_mixture_single, plot_mixture_directory

results_dir = paths['regression_results'] / 'HUXt3' / 'phase5b_hotstart'

for hs in ['default', 'hotstart']:
    print(f'=== Hot start = {hs} ===')
    results_file = results_dir / f'results_p5b_k3_meanpool_{hs}_seed42_lt12_fold0.pkl'
    plot_mixture_single(results_file)

## Hot-Start (Range × K) Sweep — Assessment

**Runs:** mean-pool set encoder, v-only, 12h, fold 0, seed 42,
`intensity_type='none'` (UNWEIGHTED). `init_spread=True` throughout; swept
initialisation range × K ∈ {2,3,4,5}. Diagnostic: per-run collapse fraction,
component separation, min-alpha (component usage), and storm-alignment of
separation.

### Range axis — a Goldilocks effect

- **Too wide / too high fails.** Every `[1,15]` range, across all K, collapsed
  almost completely (frac_collapsed 0.98–1.0) with a dead component
  (min_alpha \~0.006–0.03). Mechanism: a component initialised at Hp30≈15 sits
  where there is essentially no data (storms rarely exceed \~9), receives no
  gradient to remain, and either loses all weight or is dragged back. `[2,8]`
  (whole spread lifted, no low anchor) also collapsed heavily.
- **Moderate low-anchored ranges are healthiest.** `[1,7]` and `[1,9]` gave the
  lowest collapse with the highest min-alpha (both components genuinely used;
  K=2 min_alpha \~0.40–0.44). Anchoring the lowest component near the bulk (\~1)
  matters — starting all components high leaves none where most data lives.

### K axis — higher K adds dead components, not structure

- **K=2 gives the healthiest structure** (two genuinely-used components at the
  good ranges). As K increases (3→5), the extra components increasingly collapse
  (min_alpha → 0): the data does not support 4–5 distinct modes in what is
  essentially a bulk plus a thin tail. K=3 at [1,7] = two working components plus
  a near-dead third (min_alpha 0.06). Higher K is not beneficial here.

### Cross-cutting finding — separation is NOT storm-aligned

- Across every K and range, `collapsed_storm_frac ≈ separated_storm_frac` (often
  collapsed *higher*). The forecasts that maintain multiple modes are **not**
  disproportionately storms. Whatever separation the mixture achieves happens in
  the non-storm range and does not track the storm/quiet distinction.

### Synthesis

Hot-start initialisation **strongly controls whether the mixture separates** —
collapse fraction swings from \~6% to \~100% with the init range alone —
confirming that mild/absent multimodality in earlier (default-init) runs was an
**optimisation / failure-to-separate artefact, not a data-driven collapse**.
A Goldilocks range (moderate, low-anchored, ≈[1,7]–[1,9]) yields healthy
two-component structure. However, **no range or K setting makes the separation
storm-aligned**: components separate within the non-storm range, and storm
forecasts still show apparent collapse because no storm-range mode is ever
learned to deploy.

This isolates the two failures and motivates the next experiment:
- **Where components can sit** — fixed by hot-start (init into the storm range).
- **Whether weight stays on a storm-range component** — NOT fixed by init alone,
  because the unweighted loss (bulk-dominated, storms rare) gives no incentive.

→ **Next:** hot-start + intensity-weighted NLL. Init places a component in the
storm range; storm-weighting rewards keeping weight on it. If a genuine,
storm-aligned mode emerges, the ceiling was optimisation + incentive (fixable).
If weighting merely shifts the bulk (no storm-specific weight on the placed
component), the limit is informational — the input cannot identify storms, so no
storm mode can be deployed to the right forecasts even when one is available.

In [ ]:
# Components of the distribution — selected runs only
from storm_regression.plotting import plot_mixture_single
from pathlib import Path

results_dir = paths['regression_results'] / 'HUXt3' / 'phase5b_hotstart_ranges'

# the flagged runs (match by substring so seed/lt/fold suffix doesn't matter)
selected = [
    'p5b_hotstart_k2_1_9',    # healthiest K=2 (min_alpha 0.44) — best two-component structure
    'p5b_hotstart_k2_1_7',    # other healthy K=2 — compare to [1,9]
    'p5b_hotstart_k2_1_15',   # near-total collapse — dead-component-in-empty-tail mechanism
    'p5b_hotstart_k3_1_7',    # K=3 — does the 3rd component add structure or just die?
]

for tag in selected:
    matches = list(results_dir.glob(f'*{tag}*.pkl'))
    if not matches:
        print(f'!! no file matching {tag}')
        continue
    f = matches[0]
    print('===', f.name, '===')
    plot_mixture_single(f, save_name=f.stem)

In [ ]:
### numerical interpretation

from pathlib import Path
import numpy as np, pandas as pd
from storm_regression.results_io import load_results

results_dir = paths['regression_results'] / 'HUXt3' / 'phase5b_hotstart_ranges'

rows = []
for f in sorted(results_dir.glob('*.pkl')):
    results, config, _ = load_results(f)
    mu, alpha = np.asarray(results['mu_m']), np.asarray(results['alpha'])
    y = np.asarray(results['y_test']).ravel()
    med = np.exp(mu)
    sep = np.abs(med[:, 0] - med[:, 1])
    min_alpha = alpha.min(1)
    collapsed = (sep < 0.5) | (min_alpha < 0.05)
    rows.append({
        'run': f.stem,
        'n': len(y),
        'frac_collapsed': collapsed.mean(),
        'sep_mean': sep.mean(), 'sep_median': np.median(sep),
        'min_alpha_mean': min_alpha.mean(),
        'collapsed_storm_frac': (y[collapsed] > 4.5).mean() if collapsed.any() else np.nan,
        'separated_storm_frac': (y[~collapsed] > 4.5).mean() if (~collapsed).any() else np.nan,
        'collapsed_mean_y': y[collapsed].mean() if collapsed.any() else np.nan,
        'separated_mean_y': y[~collapsed].mean() if (~collapsed).any() else np.nan,
    })
df = pd.DataFrame(rows)
print(df.to_string(index=False))

In [ ]:
results_dir = paths['regression_results'] / 'HUXt3' / 'phase5b_hotstart_ranges'

# the flagged runs (match by substring so seed/lt/fold suffix doesn't matter)
selected = [
    'p5b_hotstart_k2_1_9',    # healthiest K=2 (min_alpha 0.44) — best two-component structure
    'p5b_hotstart_k2_1_7',    # other healthy K=2 — compare to [1,9]
    'p5b_hotstart_k2_1_15',   # near-total collapse — dead-component-in-empty-tail mechanism
    'p5b_hotstart_k3_1_7',    # K=3 — does the 3rd component add structure or just die?
]

selected_cases = [f for tag in selected for f in results_dir.glob(f'*{tag}*.pkl')]

case_df = auto_detect_and_plot_case_studies(selected_cases)

## Hot-Start + Weighted-Loss Strength Sweep (K=3 mean-pool)

**Runs:** mean-pool set encoder, K=3, v-only, 12h, fold 0, seed 42. Hot-start
init range [1, 9] fixed; step intensity-weighting swept at strength ∈ {1, 3, 10}
(storms weighted `strength`×, non-storms ×1). Hypothesis: with a component
hot-started into the storm range, a storm-emphasising loss should *reward keeping
weight on it* and produce a distinct storm component — rather than merely
shifting the bulk (the failure mode seen on the flat MLP, which had no placed
component to absorb the emphasis).

### Results by weighting strength

**w = 1 (effectively unweighted):** no storm component survives. Two widely
separated components (medians ~4 and ~2.5 — the same high-/low-non-storm split
seen without weighting), plus a third low-non-storm component that is dead
(alpha < 0.05 throughout). Consistent with earlier unweighted runs: separation
exists but sits entirely in the non-storm range.

**w = 3 (moderate — the informative result):** a **distinct storm component
forms**, centred at median ~5 with genuine, variable weight (alpha ~0.2–0.6) and
a tight spread (sigma ~0.1–0.2). Alongside it, a bulk component at median ~3.3
(alpha ~0.4–0.8, sigma ~0.1–0.45). The third component again dies (alpha < 0.05).
This demonstrates that **under an appropriately-weighted loss, and with the
component pre-placed by hot-start, a storm-range component genuinely survives and
carries meaningful weight** — the storm mode we had been unable to produce by
initialisation alone.

**w = 10 (too high):** over-constrained. The storm component is *forced* to
survive by the large penalty (all mu pinned 4.8–5.5, alpha 0.45–0.8) but is held
up against the data's pull — it "wants" to drift below 5 given its high learned
weight but the penalty prevents it. The bulk suffers correspondingly (much
reduced contribution). Clear evidence that w = 10 over-weights storms, sacrificing
overall fit to avoid storm loss.

### Key findings

- **The storm mode is recoverable.** Hot-start (places a component in the storm
  range) + moderate intensity-weighting (rewards keeping weight there) *together*
  produce a distinct storm component — neither lever achieves this alone. This
  refines the earlier "informational ceiling" reading: the storm structure is not
  absent from what the model can represent; it simply requires both correct
  initialisation and a loss that counteracts the storm rarity.
- **Weighting strength brackets a sweet spot.** w = 1 (no storm component) →
  w = 3 (healthy storm component) → w = 10 (over-constrained, bulk starved). The
  usable range sits around moderate weighting.
- **Bimodality survives throughout; only the vestigial third component dies.**
  (Note: the `frac_collapsed` summary metric — defined on the global minimum
  alpha — mis-flags these K=3 runs as "collapsed" because the third component is
  dead, even though the top two components remain cleanly separated. The metric
  should be redefined for K > 2 on the top-two-by-weight components; the raw
  distributions show clear surviving bimodality.)

### Open questions / next steps

- **Is the storm component storm-*aligned*?** Its weight varies (0.2–0.6) at
  w = 3 — the decisive check is whether that weight is elevated specifically on
  *actual* storm forecasts vs quiet ones. Component *existence* is established;
  component *targeting* (firing on the right forecasts) is the stronger claim and
  is not yet confirmed. [Compute mean alpha of the storm component on storm vs
  quiet windows.]
- **Principled weight selection.** The sweet spot motivates choosing the weight
  from the dataset's storm imbalance rather than by hand — see next section
  (inverse class frequency).
- Single seed/fold; the strength trend should be confirmed across seeds before
  being relied on.

In [ ]:
### numerical interpretation
results_dir = paths['regression_results'] / 'HUXt3' / 'phase5b_hotstart_weighted'

df_metrics = fast_batch_metrics(results_dir)
df_metrics

In [ ]:
# Components of the distribution — selected runs only
from storm_regression.plotting import plot_mixture_single
from pathlib import Path

results_dir = paths['regression_results'] / 'HUXt3' / 'phase5b_hotstart_weighted'

for w in [1, 3, 10]:
    filepath = results_dir / f'results_p5b_k3_hotstart1_9_wstep{w}_seed42_lt12_fold0.pkl'
    plot_mixture_single(filepath)

In [ ]:
case_df = auto_detect_and_plot_case_studies(results_dir)

## An Empirically-Driven Storm-Weighting Scheme

**Runs:** mean-pool set encoder, hot-start [1, 9], v-only, 12h, fold 0, seed 42,
step intensity-weighting at strength w ∈ {2, 3.28, 4}, for K=2 and K=3. The
central weight, 3.28, is the inverse class frequency (N_nonstorm/N_storm =
5818/1772) — the weight at which the storm and non-storm classes contribute
equally to the loss.

> ⚠️ K labelling to verify: the two metric tables (nominally K=2 and K=3) are
> near-identical in their aggregate/subset CRPS. Confirm each run's true K via
> `mu_m.shape[1]` before relying on the K=2-vs-K=3 comparison — the run names do
> not encode K, and K-propagation has silently failed earlier in this project.

### Skill metrics (all K, all weights)

Aggregate skill *degrades* as weight increases: all-test CRPS 0.64 (w=2) → 0.70
(w=3.28) → 0.71 (w=4); all-test PIT-KS 0.11 → 0.21 → 0.22. This is the expected
over-weighting cost: heavier storm emphasis improves the storm subset (storm CRPS
0.80 → 0.65 → 0.63; storm bias −1.15 → −0.82 → −0.79) but at rising quiet-period
cost (quiet CRPS 0.59 → 0.72 → 0.73; quiet bias +0.59 → +0.94 → +0.96). The
aggregate, dominated by the quiet bulk, worsens accordingly.

**On pure skill, w=2 is the best compromise** — best aggregate CRPS/calibration
while still substantially reducing the storm undershoot. w=3.28 and w=4 trade too
much quiet skill for their storm gains. (Overall CRPS rises modestly from ~0.61
to ~0.63–0.70; the ~0.61→0.63 part is within run-to-run fluctuation, but the
climb to 0.70 at higher weights is a real over-weighting effect.)

This is the same storm-vs-quiet **tradeoff frontier** identified on the flat MLP:
weighting relocates skill along the frontier (storms ↑, quiet ↓) rather than
lifting total skill.

### Distributional structure (component shapes)

Skill is not the only criterion — the *representational* question is whether a
distinct storm component forms. Here the picture differs from the skill ranking:

**K=2:**
- **w=2:** one active-period component (median \~4.6, weight \~0.30, sharper/lower
  sigma) and one non-active component (\~0.65 weight). >95% of forecasts remain
  bimodal (no collapse to unimodal).
- **w=3.28:** very similar but with *markedly reduced overlap* — two very distinct
  components; the active component's mu is higher and sharper (4.5–5.5), its sigma
  larger, and there is essentially zero overlap in the components' weight (alpha)
  distributions. Slightly more collapsing points, but no clear storm/quiet split
  in where collapse occurs.
- **w=4:** near-identical to 3.28.

**K=3:**
- **w=2 (the best-structured run observed):** three very clear components — a
  non-active, a low-active, and a distinct storm component — with median mu ~2.5,
  ~3.6, ~5.0 and weights ~(0.3–0.7), (0.3–0.6), (0.06–0.2) respectively. Sigma and
  alpha distributions are clean per component; negligible collapse between
  components 0 and 1. This is the clearest three-regime structure seen so far.
- **w=3.28:** collapses to an effective K=2 — the third component's weight falls
  below 0.03 throughout and its mu overlaps component 0. Components 0 and 1 remain
  clearly separated, presenting essentially as the K=2 w={3.28, 4} structure.
- **w=4:** as w=3.28, with an even sharper storm-time component.

### Reading — K=3 collapses to K=2 under the class-balance weight

The near-identical K=2 and K=3 metrics at w=3.28 and w=4 are not coincidental:
they are the signature of **K=3 reducing itself to a K=2 solution** at these
weights. The distributional inspection confirms this directly — at w=3.28/4 the
K=3 third component dies (alpha < 0.03, mu overlapping component 0), so the model
forecasts identically to K=2. The data supports only two working components at
these weights; the third is vestigial.

**The three-component structure survives only at the lighter w=2.** There, K=3
sustains three clean regimes (non-active ~2.5, low-active ~3.6, storm ~5.0). As
the weight increases toward the class balance (3.28) and beyond, the two lower
components consolidate into a single bulk, leaving an effective bulk-plus-storm
K=2 structure. So weighting strength and usable K trade off: heavier storm
emphasis simplifies the mixture toward two components.

### Skill vs structure

- **Best aggregate skill:** w=2 (all-test CRPS 0.64 vs 0.70+ at higher weights;
  higher weights over-weight the tail and degrade the quiet bulk).
- **Richest sustainable structure:** K=3 at w=2 (three interpretable regimes).
- **Sharpest two-component separation:** K=2 at w=3.28 (maximally distinct
  bulk/storm components) — but K=3 offers nothing extra here, as it collapses to
  this same two-component solution.

### Provisional take

- At the principled class-balance weight (3.28), **K=2 is sufficient** — K=3
  collapses to it. K=3 only earns its third component at lighter weighting (w=2).
- **w=2 is best for aggregate skill and for sustaining three-regime structure**;
  w=3.28/4 give sharper/committed storm components at the cost of aggregate
  calibration and (for K=3) the third component.
- The choice is a genuine tradeoff: w=2/K=3 for the richest interpretable
  distribution, w=3.28/K=2 for the most committed storm component. No single
  setting optimises structure, storm skill, and aggregate skill simultaneously —
  the storm-vs-quiet frontier again.

### Caveats / next

- Single seed/fold; confirm across seeds before locking a weight.
- **Confirm the true K of each run** (metric tables are near-identical between the
  nominal K=2 and K=3 — see warning above).
- Whether the storm component is storm-*aligned* (its weight elevated on actual
  storm windows) remains the key open check — structure is established, targeting
  is not yet confirmed.
- All on mean-pooling; the weighting should be revisited (lightly) on the final
  attention architecture rather than locked here.

In [ ]:
from storm_regression.forecast_analysis import fast_batch_metrics

results_dir = paths['regression_results'] / 'HUXt3' / 'phase5b_hotstart_empirically_weighted_k3'

df_metrics = fast_batch_metrics(results_dir)
df_metrics

In [ ]:
# Components of the distribution — selected runs only
from storm_regression.plotting import plot_mixture_single
from pathlib import Path

results_dir = paths['regression_results'] / 'HUXt3' / 'phase5b_hotstart_empirically_weighted_k3'

for w in [2, 3.28, 4]:
    print('===', 'storm weight, w =', w, '===')
    filepath = results_dir / f'results_p5b_hotstart1_9_wstep{w}_seed42_lt12_fold0.pkl'
    plot_mixture_single(filepath)

In [ ]:
## Some case studies for the K=3 case: 

results_dir = paths['regression_results'] / 'HUXt3' / 'phase5b_hotstart_empirically_weighted_k3'

case_df = auto_detect_and_plot_case_studies(results_dir)

In [ ]:
from storm_regression.forecast_analysis import fast_batch_metrics

results_dir = paths['regression_results'] / 'HUXt3' / 'phase5b_hotstart_empirically_weighted_k2'

df_metrics = fast_batch_metrics(results_dir)
df_metrics

In [ ]:
# Components of the distribution — selected runs only
from storm_regression.plotting import plot_mixture_single
from pathlib import Path

results_dir = paths['regression_results'] / 'HUXt3' / 'phase5b_hotstart_empirically_weighted_k2'

for w in [2, 3.28, 4]:
    print('===', 'storm weight, w =', w, '===')
    filepath = results_dir / f'results_p5b_hotstart1_9_wstep{w}_seed42_lt12_fold0.pkl'
    plot_mixture_single(filepath)

In [ ]:
## Some case studies for the K=2 case: 

results_dir = paths['regression_results'] / 'HUXt3' / 'phase5b_hotstart_empirically_weighted_k2'

case_df = auto_detect_and_plot_case_studies(results_dir)

## Attention Pooling

**Question:** does learned per-member weighting — the generalisation of Paper 1's
fixed 1/MAE² member weighting — extract storm-discriminating information that
uniform mean-pooling averages away?

**Setup:** set encoder with attention-weighted pooling (`se_attention=True`): a
per-member scorer produces one softmax weight per ensemble member, scoring each
member *in isolation* (weight depends only on that member's own embedding). K=2/3,
hot-start [1,9], with and without step-weighting, 12h, fold 0, seed 42. Matched to
the mean-pool runs so the only change is the pooling.

**Finding — skill:** attention **ties mean-pooling** across every matched config
(storm CRPS ~1.02 unweighted, ~0.81 at wstep2 — all within noise of mean-pool).
Learned per-member weighting provides no skill improvement over uniform averaging.

**Finding — the attention is real, not collapsed.** Inspecting the learned weights
(see plot): the attention is genuinely non-uniform — logits span ~0.5–4.1, weights
span ~0.0002–0.063 (vs uniform 0.005), and the effective number of members
attended is ~150/200, i.e. the model actively suppresses ~a quarter of the
ensemble. So the tie is *not* because attention degenerated to mean-pooling — it
learned a real, structured weighting and that weighting simply does not improve
the forecast.

**Finding — the weighting is storm-invariant.** Pooled over the full test set,
effective-N is essentially identical on storm vs quiet windows (158.5 vs 156.1 at
threshold 4.5, n=288; 159.0 vs 156.6 at threshold 6.0, n=55 — ~2.4-member gaps).
Mann-Whitney p=0.069 (n.s.); KS p=0.029 (marginal, negligible effect size, and in
the *less*-selective direction on storms). The model does not attend differently
when forecasting a storm — it applies the same member weighting regardless of
outcome. A visually apparent storm/quiet difference in a small single-batch sample
dissolved on full-test pooling and significance testing.

**Reading.** Isolated-member attention learns a sensible, storm-invariant
robustification of the ensemble (down-weight consistently-uninformative members),
which is — unsurprisingly — no better than the mean. This characterises *one*
attention mechanism thoroughly but does not establish a ceiling: this scorer judges
each member alone and structurally *cannot* ask whether a member is anomalous
relative to the ensemble. That relational question is the natural next test
(inter-member self-attention, below).

In [ ]:
from storm_regression.forecast_analysis import fast_batch_metrics

results_dir = paths['regression_results'] / 'HUXt3' / 'phase5b_attention'

df_metrics = fast_batch_metrics(results_dir)
df_metrics

In [ ]:
# Components of the distribution — selected runs only
from storm_regression.plotting import plot_mixture_single
from pathlib import Path

results_dir = paths['regression_results'] / 'HUXt3' / 'phase5b_attention'

for tag in ['nohot', 'hot1_9', 'hot1_9_wstep2']:
    print('===', 'tag', tag, '===')
    filepath = results_dir / f'results_p5b_attn_k3_{tag}_seed42_lt12_fold0.pkl'
    plot_mixture_single(filepath)

In [ ]:
# Components of the distribution — selected runs only
from storm_regression.plotting import plot_mixture_single
from pathlib import Path

results_dir = paths['regression_results'] / 'HUXt3' / 'phase5b_attention'

for tag in ['nohot', 'hot1_9', 'hot1_9_wstep2']:
    print('===', 'tag', tag, '===')
    filepath = results_dir / f'results_p5b_attn_k2_{tag}_seed42_lt12_fold0.pkl'
    plot_mixture_single(filepath)

In [ ]:
## Check how the attention is training: 
from storm_regression.plotting import plot_loss_curve

filepath = results_dir / f'results_p5b_attn_k2_hot1_9_wstep2_seed42_lt12_fold0.pkl'
plot_loss_curve(filepath)

In [ ]:
from IPython.display import Image
Image(filename='attention_weights.png') 

## Inter-Member Self-Attention

**Question:** the isolated-member attention above scores each member independently
and cannot assess a member *relative to the ensemble*. Does inter-member
self-attention — letting each member's representation be updated by attending to
all others before pooling (Set Transformer style; Lee et al. 2019) — recover
storm-discriminating structure that isolated processing missed? This is the most
expressive ensemble-processing mechanism tested, and the decisive test of whether
the ceiling is architectural or informational.

**Setup:** self-attention block(s) applied to the member embeddings before pooling
(`se_self_attention=True`, 4 heads, 1 layer), composed with both mean-pooling and
attention-pooling. Permutation invariance verified to ~5e-8 (self-attention is
equivariant; pooling makes it invariant). K=3, hot-start [1,9], wstep2, 12h, fold 0,
seed 42. Matched to the mean-pool run.

**Finding — self-attention also ties mean-pooling.** Storm CRPS 0.79 (attnpool) /
0.79 (meanpool) vs mean-pool's ~0.80; storm bias −1.13/−1.12 vs −1.14; all-test and
quiet essentially identical. The most expressive architecture available lands on
the same forecast as simple averaging.

**Finding — the two pooling variants are also identical to each other.** Self-attn
+ mean-pool and self-attn + attention-pool give the same metrics — meaning that once
self-attention has (re)contextualised the members, the downstream pooling choice
makes no difference. Had the self-attention found exploitable inter-member
structure, selective pooling on top would have mattered; it doesn't.

**Finding — the mixture structure is good regardless.** Self-attention produces the
clean three-regime mixture (low ~2, bulk ~3.3 at ~0.75 weight, storm ~5 at ~0.15
weight, tight storm sigma). So the tie is not a failure of the mixture or of
training — the model builds sensible, well-separated distributions; they simply
carry no more storm skill than mean-pool's.

**Reading — the ceiling is informational, not architectural.** Four
architecturally distinct ways of processing the velocity ensemble — flat MLP on
percentiles, mean-pool, isolated-member attention, and inter-member self-attention
— all converge on the same storm skill. Self-attention was the mechanism that
*could* ask the relational questions the others could not, and it was the test that
could have broken the pattern; it did not, despite training cleanly and building
good structure. This convergence across genuinely different architectures is what
supports the conclusion: the information distinguishing storm *magnitude* is not
recoverable from the ambient solar-wind velocity ensemble by any processing of it,
because it is not present in it — consistent with storm magnitude being controlled
largely by IMF orientation (Bz / coupling), which the velocity ensemble does not
carry.

**Caveats.** Single seed/fold; the tie should replicate across seeds before the
ceiling claim is final. Per-member features are velocity-only — velocity gradient
(vgrad) remains an untested per-member enrichment. The informational-ceiling
hypothesis is directly tested in Phase 6 (the oracle experiment: does perfect
in-window driver knowledge, including Bz/coupling, break the ceiling?).

In [ ]:
from storm_regression.forecast_analysis import fast_batch_metrics

results_dir = paths['regression_results'] / 'HUXt3' / 'phase5b_selfattn'

df_metrics = fast_batch_metrics(results_dir)
df_metrics

In [ ]:
# Components of the distribution — selected runs only
from storm_regression.plotting import plot_mixture_single
from pathlib import Path

results_dir = paths['regression_results'] / 'HUXt3' / 'phase5b_selfattn'

for tag in ['attnpool', 'meanpool']:
    print('===', 'tag', tag, '===')
    filepath = results_dir / f'results_p5b_selfattn_{tag}_k3_hot1_9_wstep2_seed42_lt12_fold0.pkl'
    plot_mixture_single(filepath)